# 增强网页爬虫（BeautifulSoup + Playwright）

## 练习目标（理念）

在课程 **Day 1「网页摘要器」** 基础上走完整链路：

1. 用 `requests` + **BeautifulSoup** 抓「静态」站点，拼 system/user messages，调用 **OpenAI** 做 Markdown 摘要
2. 发现 `https://openai.com` 这类 **JavaScript 渲染** 站点时，静态抓取失败
3. 用 **Playwright（async）** 真正开 Chromium 渲染页面，再同样交给 LLM 摘要

目标形态：给 URL → 返回可读摘要（「互联网读者文摘」）。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `.env` + `OPENAI_API_KEY` | `load_dotenv` 与密钥格式检查 |
| Chat Completions | `openai.chat.completions.create(...)` |
| system / user | `system_prompt` + `user_prompt_for` / `messages_for` |
| 静态抓取 | `class Website` + `requests.get` + BeautifulSoup |
| JS 站点增强 | 后半段 `async_playwright` 重写 `Website.create` |

## 怎么跑

1. 从课程项目根目录启动 Jupyter，并激活虚拟环境
2. 准备 `.env`：有效的 `OPENAI_API_KEY`
3. 先从上到下跑静态摘要链路（edwarddonner / cnn / anthropic）
4. 安装 Playwright 与浏览器：`pip install playwright` 后执行 `playwright install chromium`
5. 再跑异步格，对 `https://openai.com` 做增强抓取摘要

## 课前准备（可选阅读）

- 环境：仓库根目录的 `SETUP-PC.md` / `SETUP-mac.md`
- Jupyter：Shift+Enter 逐格运行；可用 `!` 跑 shell
- 命令行 / IDE / Python 基础：课程官方 guides 与 Intermediate Python 笔记本
- 故障排除：原 week1 目录下的 troubleshooting 笔记本

## 重要说明

- **先看讲座，再亲手跑**：加 `print`、改 URL、做自己的变体
- 代码会随课程更新；以笔记本当前内容为准
- 摘要是经典 GenAI 用例——想一想如何用到新闻、财报、简历等业务场景


In [ ]:
# ========== 导入：环境、HTTP、解析、展示、OpenAI ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入 requests：用 HTTP GET 拉取网页 HTML（静态抓取）
import requests
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可查询的文档树，便于抽正文
from bs4 import BeautifulSoup
# 从 IPython.display 导入 Markdown/display：在笔记本里漂亮渲染模型返回的 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端：调用云端 Chat Completions API
from openai import OpenAI

# 若本格 ImportError：到 week1 的故障排除笔记本按步骤排查依赖/环境


# 连接到 OpenAI（或 Ollama）

下一格会：

1. `load_dotenv(override=True)` 加载 `.env`
2. 读取并检查 `OPENAI_API_KEY`
3. 用 `OpenAI()` 创建客户端（默认读环境变量里的密钥）

想用免费本地 **Ollama** 时，可参考课程 README「付费 API 的免费替代方案」以及 `day1_with_ollama.ipynb`。

## 出问题怎么排

- 打开同目录/week1 的 troubleshooting 笔记本逐步诊断
- 改过环境变量后：内核菜单 →「重新启动内核并清除所有输出」，再从头运行
- API 成本：Day 1 调用量很小；也可用 Ollama（第 2 天会细讲）


In [ ]:
# ========== 加载 .env 并校验 OPENAI_API_KEY ==========

# override=True：.env 里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境读取 OpenAI 密钥；变量名必须是 OPENAI_API_KEY（常见约定）
api_key = os.getenv('OPENAI_API_KEY')

# ========== 密钥体检：缺了 / 前缀不对 / 首尾空白 ==========

# 完全没读到密钥
if not api_key:
    # 错误提示字符串保持英文原样：便于对照官方 troubleshooting 文案
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 项目密钥通常以 sk-proj- 开头；前缀不对多半拷错了 key
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# strip 后不同 → 说明首尾有空格/制表符，请求时常会 401
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 通过上述检查，先继续；真正能否调用还要看下一格 OpenAI()
    print("API key found and looks good so far!")


In [ ]:
# ========== 创建 OpenAI 客户端（默认使用环境变量中的 API Key）==========

# 无参构造：SDK 会自动读 OPENAI_API_KEY
openai = OpenAI()

# 若后面调用失败：内核 → 重新启动并清空输出，再从头跑
# 仍失败：看 week1 故障排除笔记本


# 快速预览：先对 Frontier 模型做一次最小调用

下面用最短的 `messages` 打通 `chat.completions.create`，确认密钥与网络正常，再进入网页摘要项目。


In [ ]:
# ========== 最小 Chat Completions 预览 ==========

# 发给模型的用户消息（prompt 字符串保持英文，避免改变模型行为）
message = "Hello, GPT! This is my first ever message to you! Hi!"
# 非流式调用：model 与 messages 是两个核心参数
response = openai.chat.completions.create(model="gpt-4o-mini", messages=[{"role":"user", "content":message}])
# choices[0].message.content：取出助手回复正文并打印
print(response.choices[0].message.content)


## 开始第一个项目：静态网页 → 摘要

下面用 `Website` 类封装「抓标题 + 正文」，再逐步拼 prompt 与 messages。


In [ ]:
# ========== Website 类：用 requests + BeautifulSoup 抓静态页 ==========

# 代表「一个网页」的数据对象；不熟 class 可先看 Intermediate Python 笔记本

# 有些站点会拦默认 UA；带上浏览器风格的 User-Agent 更像真人访问
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        # 保存原始 URL，便于调试
        self.url = url
        # GET 拉取 HTML 字节；headers 降低被拒概率
        response = requests.get(url, headers=headers)
        # 解析 HTML；'html.parser' 是 Python 内置解析器
        soup = BeautifulSoup(response.content, 'html.parser')
        # 取 <title>；没有标题时给占位字符串（字符串保持原样）
        self.title = soup.title.string if soup.title else "No title found"
        # 删掉 script/style/img/input，减少导航与噪音进摘要
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 抽出可见文本：换行分隔，strip 去掉多余空白
        self.text = soup.body.get_text(separator="\n", strip=True)


In [ ]:
# ========== 试抓 edwarddonner.com：看 title / text ==========

# 实例化 Website；可改成你想试的 URL
ed = Website("https://edwarddonner.com")
# 打印页面标题
print(ed.title)
# 打印清洗后的正文（可能很长）
print(ed.text)


## 提示类型（system / user）

像 GPT-4o 这类聊天模型，训练时约定了「角色 + 内容」的消息格式：

- **系统提示（system）**：任务是什么、语气如何、输出格式
- **用户提示（user）**：本轮真正要处理的内容（这里是网页正文）

后面两格会分别定义 `system_prompt` 与 `user_prompt_for(website)`。


In [ ]:
# ========== system_prompt：定「摘要助手」的角色与输出格式 ==========

# 可把最后一句改成其它语言要求做实验；发给模型的英文指令本身不要翻译
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."


In [ ]:
# ========== user_prompt_for：把标题 + 正文拼进 user 消息 ==========

def user_prompt_for(website):
    # 先告诉模型「正在看哪个站点标题」
    user_prompt = f"You are looking at a website titled {website.title}"
    # 任务说明保留英文：要求短摘要，若有新闻/公告也一并总结
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    # 追加抓到的正文文本
    user_prompt += website.text
    # 返回完整 user 字符串，供 messages 使用
    return user_prompt


In [ ]:
# ========== 预览：打印拼好的 user prompt（可能很长）==========

print(user_prompt_for(ed))


## 消息（messages）结构

OpenAI Chat Completions（以及很多兼容 API）期望：

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

下面两格先用一个「毒舌助手算 2+2」的小例子，确认 system+user 一起生效。


In [ ]:
# ========== 示例 messages：system 定人设，user 提问 ==========

messages = [
    # system：角色/语气（字符串保持英文）
    {"role": "system", "content": "You are a snarky assistant"},
    # user：具体问题
    {"role": "user", "content": "What is 2 + 2?"}
]


In [ ]:
# ========== 用上面的 messages 调用 gpt-4o-mini ==========

# 把 messages 整表传给 API
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
# 打印助手回复
print(response.choices[0].message.content)


## 为网页摘要组装 messages

把 `system_prompt` 与 `user_prompt_for(website)` 收成一个函数，后面 `summarize` 直接用。


In [ ]:
# ========== messages_for：网站对象 → 标准两段式 messages ==========

def messages_for(website):
    # 返回与上面示例相同结构：system + user
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]


In [ ]:
# ========== 试看 messages_for(ed) 的结构 ==========

# 返回的是 list[dict]；可再换别的 Website 实例试
messages_for(ed)


## 串起来：URL → Website → messages → 摘要

OpenAI 侧调用本身很短；复杂点在「抓取与拼 prompt」。


In [ ]:
# ========== summarize：抓站 + 调用 Chat Completions，返回摘要字符串 ==========

def summarize(url):
    # 用静态 Website 类抓取该 URL
    website = Website(url)
    # 非流式创建聊天补全；model / messages 保持原参数
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    # 取出第一条 choice 的助手文本
    return response.choices[0].message.content


In [ ]:
# ========== 试跑：摘要 edwarddonner.com（原始字符串）==========

summarize("https://edwarddonner.com")


In [ ]:
# ========== display_summary：用 Markdown 渲染摘要 ==========

def display_summary(url):
    # 先拿到摘要字符串
    summary = summarize(url)
    # 在 Jupyter 里按 Markdown 显示
    display(Markdown(summary))


In [ ]:
# ========== 展示摘要：edwarddonner.com ==========

display_summary("https://edwarddonner.com")


# 再试更多网站

注意：上面的 **静态** 方案只适合「服务器直接返回完整 HTML」的站点。

- **JavaScript 渲染**（如很多 React 站）用 `requests` 往往拿到空壳 → 正文很少或没有
- 社区贡献里有 Selenium / Playwright 方案（本笔记本后半段就是 Playwright）
- 部分受 CloudFront 等保护的站可能返回 **403**（感谢 Andy J 反馈）

许多内容站仍然可以直接工作——下面试 CNN 与 Anthropic。


In [ ]:
# ========== 展示摘要：cnn.com ==========

display_summary("https://cnn.com")


In [ ]:
# ========== 展示摘要：anthropic.com ==========

display_summary("https://anthropic.com")


## 商业应用

你刚走通了调用 **Frontier Model** 云端 API 的路径。除了以后自己训模型，课程里会大量使用这类 API。

更具体地说，这里做的是 **总结（summarization）**——经典 GenAI 用例：新闻、财报、简历/求职信……场景很多。想一想能否在你自己的业务里做个小原型。

## 动手练习

用下一格做自己的小例子（仍可围绕摘要）。例如：粘贴一封邮件正文，让模型建议简短主题行——这就是商务邮箱助手常见能力。


In [ ]:
# ========== 学生练习脚手架：自己拼 prompt → messages → 调用 ==========

# 第 1 步：创建提示（下面是占位，请改成你的业务文案）
system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# 第 2 步：组装 messages 列表（应类似 [{"role":"system",...},{"role":"user",...}]）
messages = [] # fill this in

# 第 3 步：调用 OpenAI（补全右侧表达式，可参考前面的 summarize）
response =

# 第 4 步：打印结果（例如 print(response.choices[0].message.content)）


## 额外练习：JS 站点与浏览器自动化

若试 `display_summary("https://openai.com")`，静态方案常常失败——OpenAI 官网大量依赖 JavaScript 渲染。

常见解法：

- **Selenium** / **Playwright**：后台启动真实浏览器，等页面渲染完再取 DOM
- 有经验可直接改 `Website` 类；社区贡献文件夹里也有同学的 Selenium 示例

本笔记本接下来用 **Playwright async API** 增强抓取。


# 分享你的代码

欢迎把改进提交到社区贡献文件夹（Pull Request）。若不熟 git，可让 GPT 逐步指导提 PR。

专业提示：分享前可用「编辑 → 清除所有输出」得到更干净的笔记本（本教学注释任务除外，不要清别人的 outputs）。

参考说明：  
https://chatgpt.com/share/677a9cb5-c64c-8012-99e0-e06e88afd293


In [ ]:
# ========== 导入 Playwright：同步/异步 API + 异步工具 ==========

# sync_playwright：同步入口（本格导入备用；后面主流程用 async）
from playwright.sync_api import sync_playwright
# time：通用计时/休眠（本格导入保留；重试里实际用 asyncio.sleep）
import time 
# asyncio：异步睡眠等，配合 async Website.create 重试
import asyncio
# async_playwright：在 Jupyter 里用 await 驱动浏览器更自然
from playwright.async_api import async_playwright


In [ ]:
# （空单元格占位）下一格是 Playwright 版 Website + 异步 summarize / display_summary


In [ ]:
# ========== Playwright 增强版 Website：渲染后再抽正文 ==========

class Website:
    def __init__(self, url):
        # 保存 URL；title/text 稍后在 initialize 里填充
        self.url = url
        self.title = None
        self.text = None

    @classmethod
    async def create(cls, url):
        # 工厂方法：构造实例后 initialize；失败则有限次重试
        website = cls(url)
        retries = 3  # Add retry logic
        for attempt in range(retries):
            try:
                # 打开浏览器、导航、解析正文
                await website.initialize()
                return website
            except TimeoutError as e:
                # 最后一次仍超时则把异常抛给调用方
                if attempt == retries - 1:  # Last attempt
                    raise
                # 提示第几次失败（英文 f-string 保持原样）
                print(f"Attempt {attempt + 1} failed, retrying...")
                # 重试前等待 2 秒，给网络/挑战页一点时间
                await asyncio.sleep(2)  # Wait between retries

    async def initialize(self):
        # async with：退出时自动关闭 Playwright 驱动资源
        async with async_playwright() as p:
            # 无头 Chromium；若干参数降低「被识别为自动化」的概率
            browser = await p.chromium.launch(
                headless=True,
                args=[
                    '--disable-blink-features=AutomationControlled',
                    '--disable-dev-shm-usage',
                    '--no-sandbox'
                ]
            )
            
            # 新上下文：自定义 UA、视口、启用 JS、绕过 CSP、附加请求头
            context = await browser.new_context(
                user_agent='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
                viewport={'width': 1920, 'height': 1080},
                java_script_enabled=True,
                bypass_csp=True,  # Bypass Content Security Policy
                extra_http_headers={
                    'Accept-Language': 'en-US,en;q=0.9',
                    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8'
                }
            )
            
            # 在该上下文中打开新标签页
            page = await context.new_page()
            
            try:
                # 导航到目标 URL；超时 90s（毫秒）
                await page.goto(self.url, timeout=90000)  # 90 second timeout
                
                # 优先等 <main>；失败则退回到 networkidle + body 可见（应对 Cloudflare 等）
                try:
                    # 先等实际内容容器
                    await page.wait_for_selector('main', timeout=10000)
                except:
                    # 未找到 main：再等网络空闲与 body
                    await page.wait_for_load_state('networkidle', timeout=30000)
                    await page.wait_for_selector('body', state='visible', timeout=30000)
                
                # 渲染完成后取标题与完整 HTML
                self.title = await page.title()
                content = await page.content()
                
                # 同样用 BeautifulSoup 清 script/style 等，再抽正文
                soup = BeautifulSoup(content, 'html.parser')
                for irrelevant in soup.find_all(["script", "style", "img", "input"]):
                    irrelevant.decompose()
                self.text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""
                
            finally:
                # 无论成功失败都关闭浏览器，避免僵尸进程
                await browser.close()

# ========== 异步 summarize：工厂创建 Website 后再调 OpenAI ==========

async def summarize(url):
    # await 工厂方法，拿到已填充 title/text 的实例
    website = await Website.create(url)
    # 聊天补全仍用同步 SDK 调用（messages_for 复用前面定义）
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for(website)
    )
    return response.choices[0].message.content

# ========== 异步 display_summary：摘要 + Markdown 展示 ==========

async def display_summary(url):
    summary = await summarize(url)
    display(Markdown(summary))

# ========== 用法：对 openai.com 做增强抓取摘要（Jupyter 顶层 await）==========

await display_summary("https://openai.com")


In [ ]:
# （空单元格占位）增强爬虫主流程已在上一格；可在此继续试验其它 URL
